# Phase 7 — Streamlit UI
Viewer over `artifacts/reports/app_smoke_report.json`. The app itself is `app/streamlit_app.py`; launch it with
`streamlit run app/streamlit_app.py`. Nothing is computed here.

In [1]:
import sys
from pathlib import Path
import pandas as pd
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs" / "base.yaml").is_file())
sys.path.insert(0, str(root))
from src import ui_support as ui
from src.utils import read_json
cfg = ui.get_config()
rp = cfg["_abs"]["reports_dir"] / "app_smoke_report.json"
REP = read_json(rp) if rp.is_file() else None
print("app smoke test:", REP["status"] if REP else "NOT RUN (python scripts/smoke_test_app.py --serve-check)")
opts = ui.discover_models(cfg)
display(pd.DataFrame([{"model": o.key, "label": o.label, "available": o.available, "reason": o.reason or "-"} for o in opts]) if opts else pd.DataFrame({"note": ["no frozen selection found"]}))

app smoke test: PASS


,model,label,available,reason
0,ft_hardneg_dense,Fine-tuned (+ mined dense hard negatives),True,-
1,baseline,Baseline MiniLM-L6 (pretrained),True,-


## 1. What the app does

- **Inputs:** a text query and a top-k slider (1–50). Example queries fill the box in one click.
- **Each result:** rank, title, **paper ID** (SciFact corpus ID, for citation), snippet, **cosine similarity**, and an expandable full abstract.
- **Graceful states:** empty query → prompt; clipped/long queries → warnings; load failures → readable error, never a traceback.
- **Models:** the frozen fine-tuned model by default. The pretrained baseline (and a side-by-side comparison) appears only if its index/embeddings exist.
- **Performance:** each model loads once per process (`st.cache_resource`), with one warm-up query; the baseline loads only when selected.

## 2. Headless smoke test (real app script, real package; not a browser)

In [2]:
if REP:
    display(pd.DataFrame(REP["steps"])[["step", "ok", "seconds", "detail"]])
    print("method:", REP["method"]); print("models offered:", REP.get("models_offered")); print("manual check pending:", REP.get("manual_check_pending"))

,step,ok,seconds,detail
0,initial_load_no_exception,True,12.71,title='🔎 Scientific Paper Search (SciFact)' er...
1,empty_query_prompts_user,True,0.00,results=0
2,query_returns_default_top_k,True,0.14,results=10 expected=10
3,slider_changes_result_count,True,0.06,results=3 expected=3
4,whitespace_query_is_graceful,True,0.03,results=0
5,example_button_fills_query_and_searches,True,0.10,query='Wnt signaling in embryonic stem cell di...
6,switch_to_baseline_returns_results,True,1.01,baseline results=10
7,compare_side_by_side_shows_both_lists,True,0.09,results=20 expected=20
8,streamlit_run_server_health,True,2.85,health='ok' index_http=200 port=8599


method: streamlit.testing.v1.AppTest (headless; not a browser)
models offered: [{'available': True, 'key': 'ft_hardneg_dense'}, {'available': True, 'key': 'baseline'}]
manual check pending: Open `streamlit run app/streamlit_app.py` in a browser and visually confirm layout (cannot be automated here).


## 3. Launch (PowerShell, repo root)
```powershell
streamlit run app/streamlit_app.py
```
The first page load takes roughly 20 s while the model and index load; later searches take tens of milliseconds (see Phase 6 benchmark).

## 4. Caveats
- The smoke test drives the app script headlessly; visual layout still needs a human look in a browser.
- Scores are cosine similarities, not probabilities; results reflect similarity to 5,183 SciFact abstracts only.
- Research prototype; not medical or clinical advice.